In [11]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np

from common import (
    CASE_NAMES,
    get_dataset_sort_key,
    INDEX_ORDER,
    index_colors,
    index_hatches,
    transform_duckdb_index_name,
    transform_pgvector_index_name,
    apply_style,
    format_dataset_label,
    PLOT_DPI,
    LABEL_FONTSIZE,
    TICK_FONTSIZE,
    X_TICK_FONTSIZE,
    BAR_LABEL_FONTSIZE,
    FONT_COLOR,
    TICK_FONTS_COLOR,
    BAR_TEXT_COLOR,
)

# ---------------------------------------------------------------------------
# 1. Load all result JSONs from both DuckDB and pgvector directories
# ---------------------------------------------------------------------------
records = []

# Load DuckDB results
duckdb_results_dir = pathlib.Path("../experiments/results/index_creation/DuckDB")
duckdb_json_files = sorted(duckdb_results_dir.glob("*.json"))

for path in duckdb_json_files:
    with open(path) as f:
        data = json.load(f)
    for entry in data.get("results", []):
        metrics = entry["metrics"]
        task_cfg = entry["task_config"]
        db_cfg = task_cfg["db_config"]
        db_case_cfg = task_cfg["db_case_config"]
        case_cfg = task_cfg["case_config"]

        # Parse db_label JSON to get global_version
        db_label = json.loads(db_cfg.get("db_label", "{}"))
        global_version = db_label.get("global_version", None)

        index_name = transform_duckdb_index_name(db_case_cfg, global_version)

        records.append({
            "optimize_duration": metrics["optimize_duration"],
            "index": index_name,
            "case_id": case_cfg["case_id"],
            "threads": db_cfg["duckdb_threads"],
            "global_version": global_version,
            "db_type": "DuckDB",
        })

# Load pgvector results
pgvector_results_dir = pathlib.Path("../experiments/results/index_creation/PgVector")
pgvector_json_files = sorted(pgvector_results_dir.glob("*.json"))

for path in pgvector_json_files:
    with open(path) as f:
        data = json.load(f)
    for entry in data.get("results", []):
        metrics = entry["metrics"]
        task_cfg = entry["task_config"]
        db_cfg = task_cfg["db_config"]
        db_case_cfg = task_cfg["db_case_config"]
        case_cfg = task_cfg["case_config"]

        index_name = transform_pgvector_index_name(db_case_cfg)

        # pgvector uses max_parallel_workers in db_case_config instead of duckdb_threads in db_config
        threads = db_case_cfg.get("max_parallel_workers", None)

        records.append({
            "optimize_duration": metrics["optimize_duration"],
            "index": index_name,
            "case_id": case_cfg["case_id"],
            "threads": threads,
            "global_version": None,  # pgvector doesn't have global_version
            "db_type": "PgVector",
        })

print(f"Loaded {len(records)} results from {len(duckdb_json_files)} DuckDB files and {len(pgvector_json_files)} pgvector files")
records[:3]

ImportError: cannot import name 'apply_style' from 'common' (/Users/user/Code/vss-benchmarks/analysis/common.py)

In [ ]:
# ---------------------------------------------------------------------------
# 2. Case ID → human-readable dataset name
# ---------------------------------------------------------------------------

for r in records:
    r["dataset"] = CASE_NAMES.get(r["case_id"], f"Unknown (case_id={r['case_id']})")

unique_datasets = set(r["dataset"] for r in records)
datasets = sorted(unique_datasets, key=get_dataset_sort_key)

In [ ]:
# ---------------------------------------------------------------------------
# 3. Plot: one subplot per dataset, bars grouped by (index, threads, global_version)
# ---------------------------------------------------------------------------
from collections import defaultdict
import re
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

apply_style()

# Group records: dataset → thread_count → index_type → list of durations
grouped: dict[str, dict[int, dict[str, list[float]]]] = {
    ds: {1: defaultdict(list), 14: defaultdict(list)} for ds in datasets
}

for r in records:
    threads = r["threads"]
    if threads is not None:
        grouped[r["dataset"]][threads][r["index"]].append(r["optimize_duration"])

# Get all unique index types across all datasets and thread counts
all_index_types = set()
for dataset in datasets:
    for threads in [1, 14]:
        all_index_types.update(grouped[dataset][threads].keys())
all_index_types = [idx for idx in INDEX_ORDER if idx in all_index_types]
n_index_types = len(all_index_types)

# Create 2 subplots: one for 1T, one for 14T (stacked vertically)
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharey=False)
axes = axes.flatten()  # Flatten to make indexing easier

bar_width = 0.45  # Width of each bar
group_width = n_index_types * bar_width + 0.2  # Total width of a dataset group
group_spacing = 0.4  # Space between dataset groups

# First, collect all durations from graph 1 (Threads = 1) to determine shared y-axis limits
graph1_durations = []
for dataset in datasets:
    thread_data = grouped[dataset][1]
    for index_type in all_index_types:
        durations = thread_data.get(index_type, [])
        graph1_durations.extend(durations)

# Calculate shared y-axis limit from graph 1
if graph1_durations:
    shared_max_y = max(graph1_durations) * 1.15
else:
    shared_max_y = 100

for ax_idx, target_threads in enumerate([1, 14]):
    ax = axes[ax_idx]
    x_base = 0
    x_labels = []
    all_durations_for_ax = []

    for dataset in datasets:
        thread_data = grouped[dataset][target_threads]

        # Skip if no data for this thread count
        if not any(thread_data.values()):
            continue

        group_start = x_base
        for idx_idx, index_type in enumerate(all_index_types):
            x = group_start + idx_idx * bar_width

            durations = thread_data.get(index_type, [])
            if durations:
                mean_dur = np.mean(durations)
                all_durations_for_ax.extend(durations)
                color = index_colors.get(index_type, "#808080")
                hatch = index_hatches.get(index_type, "")
                ax.bar(x, mean_dur, bar_width, color=color, hatch=hatch,
                       label=index_type if dataset == datasets[0] else "")

                # Add value label on top of bar with offset
                offset = max(0.05, mean_dur * 0.03)
                ax.text(x, mean_dur + offset, f"{mean_dur:.1f}",
                       ha="center", va="bottom", fontsize=BAR_LABEL_FONTSIZE, color=BAR_TEXT_COLOR)

        # Use detailed dataset label for x-axis
        dataset_label = format_dataset_label(dataset)
        x_labels.append(dataset_label)
        x_base += group_width + group_spacing

    # Set title
    ax.set_title(f"{target_threads} Threads/Workers", fontsize=LABEL_FONTSIZE + 1, color=FONT_COLOR)

    # Y label for all plots
    ax.set_ylabel("Index Construction Duration (s)", fontsize=LABEL_FONTSIZE, color=FONT_COLOR)

    # Set x-ticks at the center of each dataset group
    group_centers = []
    x_base = 0
    for dataset in datasets:
        if any(grouped[dataset][target_threads].values()):
            center = x_base + (n_index_types - 1) * bar_width / 2
            group_centers.append(center)
            x_base += group_width + group_spacing

    ax.set_xticks(group_centers)
    ax.set_xticklabels(x_labels, fontsize=X_TICK_FONTSIZE, color=TICK_FONTS_COLOR)

    # Grids (horizontal only)
    ax.yaxis.grid(True, linestyle='-', linewidth=0.6, color='gray', alpha=0.2)
    ax.set_axisbelow(True)

    # Spine adjustments (remove top and right)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

    # Y-axis limits and locator (use same limits as graph 1)
    ax.set_ylim(0, shared_max_y)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.tick_params(axis='y', colors=TICK_FONTS_COLOR)
    ax.tick_params(axis='both', length=0)
    ax.tick_params(axis='x', pad=5)  # Increase spacing between x-axis labels and axis

# Shared legend - create handles for all index types
legend_handles = []
legend_labels = []
for index_type in all_index_types:
    color = index_colors.get(index_type, "#808080")
    hatch = index_hatches.get(index_type, "")
    legend_handles.append(Patch(facecolor=color, hatch=hatch, edgecolor='black', linewidth=0.5))
    legend_labels.append(index_type)

fig.legend(legend_handles, legend_labels, loc="upper center", ncols=2,
          frameon=False, fontsize=LABEL_FONTSIZE, bbox_to_anchor=(0.5, 1.04))

fig.tight_layout(rect=[0, 0, 1, 0.80])
# plt.show()
plt.savefig("index_creation.pdf", dpi=PLOT_DPI, bbox_inches='tight')